# mini RAG — from scratch, one PDF

Run each cell top to bottom. Retrieval is done by hand with numpy so nothing is hidden.

**Before you start:** make sure Ollama is running (`ollama list` in PowerShell shows `llama3.1:8b`).

In [ ]:
# Cell 1 - install deps (run once, then comment it out)
# !pip install pypdf sentence-transformers numpy ollama

In [1]:
# Cell 2 - imports + config
import numpy as np
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
import ollama

EMBED_MODEL = "BAAI/bge-small-en-v1.5"
LLM_MODEL   = "llama3.1:8b"
CHUNK_WORDS = 200
OVERLAP     = 40
TOP_K       = 10

# Windows path: use a raw string (r"...") so backslashes are not escaped,
# or just use forward slashes.
PDF_PATH = r"../data/mathematics-10-03801-v2.pdf"

In [2]:
# Cell 3 - load + chunk
def load_pdf(path):
    reader = PdfReader(path)
    return "\n".join(page.extract_text() or "" for page in reader.pages)

def chunk_text(text, chunk_words=CHUNK_WORDS, overlap=OVERLAP):
    words = text.split()
    chunks, i = [], 0
    step = chunk_words - overlap        # overlap keeps ideas that straddle a cut
    while i < len(words):
        chunks.append(" ".join(words[i:i + chunk_words]))
        i += step
    return chunks

text   = load_pdf(PDF_PATH)
chunks = chunk_text(text)
print(f"{len(chunks)} chunks")
print(chunks[0][:300])   # eyeball the first chunk

67 chunks
Citation: Sobolev, K.; Ermilov, D.; Phan, A.-H.; Cichocki, A. PARS: Proxy-Based Automatic Rank Selection for Neural Network Compression via Low-Rank Weight Approximation. Mathematics 2022, 10, 3801. https://doi.org/10.3390/ math10203801 Academic Editors: Liang Zou, Liang Zhao and Yonghui Xu Received


In [3]:
# Cell 4 - embed (first run downloads the model, ~90 MB)
def embed(model, texts):
    # normalize_embeddings=True -> every vector has length 1
    # -> a dot product between two of them IS their cosine similarity
    return model.encode(texts, normalize_embeddings=True, convert_to_numpy=True)

model = SentenceTransformer(EMBED_MODEL)
chunk_vecs = embed(model, chunks)
print(chunk_vecs.shape)   # (n_chunks, embedding_dim)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

(67, 384)


In [4]:
# Cell 5 - retrieve + generate
def retrieve(query, k=TOP_K):
    q = embed(model, [query])[0]          # (dim,)
    scores = chunk_vecs @ q               # one cosine score per chunk
    top = np.argsort(scores)[::-1][:k]
    return [(chunks[i], float(scores[i])) for i in top]

PROMPT = """You are a careful assistant. Answer the question using ONLY the context below.
If the answer is not in the context, reply exactly: "Not in the document."

Context:
{context}

Question: {question}
Answer:"""

def answer(query, hits):
    context = "\n\n---\n\n".join(t for t, _ in hits)
    resp = ollama.chat(
        model=LLM_MODEL,
        messages=[{"role": "user", "content": PROMPT.format(context=context, question=query)}],
        options={"temperature": 0.1},
    )
    return resp["message"]["content"]

def ask(query, show_chunks=False):
    hits = retrieve(query)
    if show_chunks:                       # your #1 debugging tool
        for t, s in hits:
            print(f"  [{s:.3f}] {t[:120]}...")
        print()
    return answer(query, hits)

In [5]:
# Cell 6 - ask away
print(ask("What problem does this paper address?", show_chunks = True))

  [0.523] S + T)× R parameters. Shorthand notation: ˆK = JA, B, CK. ≈K A B C [S×T ×D2] [S×R] [R×T ] [D2×R] ≈K GU V [S×T ×D2] [S×R1...
  [0.523] 2020; Springer: Berlin/Heidelberg, Germany, 2020; pp. 639–654. 45. Zacharov, I.; Arslanov, R.; Gunin, M.; Stefonishin, D...
  [0.522] the number of iterations despite the metric drop at iteration 90. This drop meant that at iteration 90, the GP’s surroga...
  [0.522] with calibration of the BN statistics. The X-axis shows the CP rank. Note: for the ﬁrst decomposed layer, layer1.1.conv1...
  [0.519] manuscript. Funding: This work was partly supported by Ministry of Science and Higher Education grant No. 075-10-2021-06...
  [0.516] ILSVRC-12 validation dataset. The↓ FLOPs column denotes the compression of the computations; the ∆ top-1 and ∆ top-5 col...
  [0.515] adjacent layers had 2–3 times lower compression ratios. In contrast, using BN calibration resulted in a better compressi...
  [0.510] which depends on the depth of the DNN; the higher th

In [6]:
print(ask("Summarize the abstract", show_chunks=True))

  [0.603] the in International Conference on Learning Representations (ICLR), San Diego, CA, USA, 7–9 May 2015. 23. Zagoruyko, S.;...
  [0.587] on Learning Representations, ICLR 2016, San Juan, Puerto Rico, 2–4 May 2016. 19. Gusak, J.; Kholyavchenko, M.; Ponomarev...
  [0.582] adjacent layers had 2–3 times lower compression ratios. In contrast, using BN calibration resulted in a better compressi...
  [0.575] Method ↓ FLOPs ∆ Top-1, %, w/o f-t ∆ Top-1, %, w f-t N total Uniform [5] 2 −12.7 −0.2 20 ENC-Inf [5] −2.9 −0.1 20 ENC-Mo...
  [0.574] Analytic Solution of Fully-Observed Variational Bayesian Matrix Factorization. J. Mach. Learn. Res. 2013, 14, 1–37. 28. ...
  [0.567] 1 Ich Ich Och D 1 R R (c) 11 Ich Ich D D R1 R2 R1 1 1 R2 Och (d) Figure 2. Visualization of different convolutional laye...
  [0.558] Citation: Sobolev, K.; Ermilov, D.; Phan, A.-H.; Cichocki, A. PARS: Proxy-Based Automatic Rank Selection for Neural Netw...
  [0.554] 3.22 - −0.04 CPD-EPC+PARS (Ours) LR 3.22 −0.30 −0.04

In [7]:
hits = retrieve("What problem does this paper address?", k=10)
for i, (chunk, score) in enumerate(hits):
    print(f"\n--- Rank {i+1} | Score: {score:.3f} ---")
    print(chunk[:200])


--- Rank 1 | Score: 0.523 ---
S + T)× R parameters. Shorthand notation: ˆK = JA, B, CK. ≈K A B C [S×T ×D2] [S×R] [R×T ] [D2×R] ≈K GU V [S×T ×D2] [S×R1] [R2×T ][R1×R2×D2] Figure A3. Decomposition of a third-order tensor into (left)

--- Rank 2 | Score: 0.523 ---
2020; Springer: Berlin/Heidelberg, Germany, 2020; pp. 639–654. 45. Zacharov, I.; Arslanov, R.; Gunin, M.; Stefonishin, D.; Bykov, A.; Pavlov, S.; Panarin, O.; Maliutin, A.; Rykovanov, S.; Fedorov, M. 

--- Rank 3 | Score: 0.522 ---
the number of iterations despite the metric drop at iteration 90. This drop meant that at iteration 90, the GP’s surrogate function put the maximum EI value at a worse point than its predecessors. Thi

--- Rank 4 | Score: 0.522 ---
with calibration of the BN statistics. The X-axis shows the CP rank. Note: for the ﬁrst decomposed layer, layer1.1.conv1 (left subﬁgure), the results of the Single and Joint settings coincide. (a) Mea

--- Rank 5 | Score: 0.519 ---
manuscript. Funding: This work was partly

In [8]:
def rewrite_query(question, context_hint=None):
    
    # if no hint given, use the first chunk as domain context
    if context_hint is None:
        context_hint = chunks[0][:300]
    
    resp = ollama.chat(
        model=LLM_MODEL,
        messages=[{"role": "user", "content": f"""
Here is an excerpt from the paper:
\"\"\"{context_hint}\"\"\"

Rephrase the question into exactly 3 variants using technical 
vocabulary from this paper — words like the ones in the excerpt above.
Output exactly 3 lines. Each line is one question.
No numbering. No explanation. No preamble. Just 3 questions.

Question: {question}
"""}],
        options={"temperature": 0.3}  # lower temp = follows instructions better
    )
    lines = resp["message"]["content"].strip().split("\n")
    # filter out preamble lines — keep only lines ending with "?"
    questions = [l.strip() for l in lines if l.strip().endswith("?")]
    return questions[:3]  # cap at 3

# test
rewrites = rewrite_query("What problem does this paper address?")
for r in rewrites:
    print(f"→ {r}")

→ What is the primary challenge that PARS aims to overcome in neural network compression?
→ How does the paper address the issue of low-rank weight approximation in neural network compression?
→ What is the main objective of the PARS algorithm in terms of automatic rank selection for neural network compression?


In [9]:
def retrieve_multi(queries, k=TOP_K):
    seen = set()
    results = []
    
    for query in queries:
        hits = retrieve(query, k=k)
        for chunk, score in hits:
            # deduplicate by first 50 chars — same chunk can appear
            # in multiple queries' results, only keep it once
            key = chunk[:50]
            if key not in seen:
                seen.add(key)
                results.append((chunk, score))
    
    # re-sort merged results by score, take top k overall
    results.sort(key=lambda x: x[1], reverse=True)
    return results[:k]

def ask_with_rewriting(question, show_chunks=False):
    print(f"Original question: {question}")
    
    # step 1 — rewrite
    rewrites = rewrite_query(question)
    all_queries = [question] + rewrites  # include original too
    
    print(f"\nRewritten into {len(rewrites)} variants:")
    for r in rewrites:
        print(f"  → {r}")
    
    # step 2 — retrieve across all variants
    hits = retrieve_multi(all_queries, k=TOP_K)
    
    if show_chunks:
        print(f"\nTop chunks after merging:")
        for i, (chunk, score) in enumerate(hits):
            print(f"\n  Rank {i+1} | Score: {score:.3f}")
            print(f"  {chunk[:150]}...")
    
    # step 3 — generate as before
    return answer(question, hits)  # note: original question goes to LLM

In [10]:
# The question that previously failed
result = ask_with_rewriting(
    "What problem does this paper address?",
    show_chunks=True
)
print("\n--- ANSWER ---")
print(result)

Original question: What problem does this paper address?

Rewritten into 3 variants:
  → What is the primary objective of the PARS method in the context of neural network compression?
  → How does the low-rank weight approximation technique contribute to the solution of the neural network compression problem?
  → What is the significance of automatic rank selection in the process of neural network compression via low-rank weight approximation?

Top chunks after merging:

  Rank 1 | Score: 0.873
  authors of a recent paper [28] found out that the pre-ﬁne-tuning accuracy, which is used in most methods to accelerate the iteration of the rank searc...

  Rank 2 | Score: 0.852
  and solves it through Bayesian optimization. This study discovered that a low-rank weight decomposition of DNN weights adversely affects its feature d...

  Rank 3 | Score: 0.839
  Method ↓ FLOPs ∆ Top-1, %, w/o f-t ∆ Top-1, %, w f-t N total Uniform [5] 2 −12.7 −0.2 20 ENC-Inf [5] −2.9 −0.1 20 ENC-Model [5] −3.5 −0.

In [11]:
# Each entry: your question + the chunk_id that contains the answer
# chunk_id format: index of the chunk in your chunks list (0-based)
# Find the right index by scanning chunks manually (see helper below)

# Helper to find which chunk contains a phrase
def find_chunk(phrase):
    for i, chunk in enumerate(chunks):
        if phrase.lower() in chunk.lower():
            print(f"chunk {i}: ...{chunk[max(0, chunk.lower().find(phrase.lower())-50):chunk.lower().find(phrase.lower())+100]}...")
    
# Example: find which chunk talks about "proxy"
find_chunk("VGG")

chunk 1: ...presentative NNs, including ResNet-18, ResNet-56, VGG-16, and AlexNet. We obtain a 3× FLOP reduction with almost no loss of accuracy for ILSVRC-2012 R...
chunk 2: ...tion with an accuracy improvement for ILSVRC-2012 VGG-16. Keywords: convolutional neural network acceleration; deep learning; low-rank tensor decompos...
chunk 3: ...SVD [5]), several DNN architectures (AlexNet [6], VGG-16 [7], ResNet-18, and ResNet-56 [8]), and two datasets (the CIF AR-10[9] and ILSVRC-2012 [10] d...
chunk 4: ...SVD [5]), several DNN architectures (AlexNet [6], VGG-16 [7], ResNet-18, and ResNet-56 [8]), and two datasets (the CIF AR-10[9] and ILSVRC-2012 [10] d...
chunk 28: ...pression methods by compressing ResNet-18 [8] and VGG-16 [7] after training on the ILSVRC-2012 [10] dataset. Thirdly, we compare our results with thos...
chunk 29: ... had top1 and top5 accuracies of 69.76 and 89.08, VGG-16 with BN had top1 and top5 accuracies of 73.36 and 91.52, and AlexNet had top1 and top5 accura...
ch

In [21]:
import json
eval_set = json.load(open("../eval/eval_set.json"))
print(f"Eval set size: {len(eval_set)} questions")

Eval set size: 10 questions


In [22]:
def recall_at_k(retrieved_indices, relevant_indices, k):
    """
    Did any relevant chunk appear in the top-k results?
    Returns 1.0 (hit) or 0.0 (miss)
    """
    top_k = set(retrieved_indices[:k])
    relevant = set(relevant_indices)
    return 1.0 if top_k & relevant else 0.0   # & = intersection

def reciprocal_rank(retrieved_indices, relevant_indices):
    """
    What rank was the first relevant chunk?
    Rewards finding it higher up.
    """
    relevant = set(relevant_indices)
    for rank, idx in enumerate(retrieved_indices, start=1):
        if idx in relevant:
            return 1.0 / rank
    return 0.0   # not found at all

In [23]:
def get_retrieved_indices(question, k=10):
    """Retrieve and return chunk indices instead of chunk text"""
    hits = retrieve(question, k=k)
    # find the index of each retrieved chunk in the chunks list
    indices = []
    for chunk, score in hits:
        for i, c in enumerate(chunks):
            if c == chunk:
                indices.append(i)
                break
    return indices

def run_eval(eval_set, k=10, use_rewriting=False):
    recall_scores = []
    mrr_scores = []
    
    print(f"Running eval | k={k} | query_rewriting={use_rewriting}\n")
    print(f"{'Question':<50} {'Recall@k':>10} {'RR':>8}")
    print("-" * 70)
    
    for item in eval_set:
        question = item["question"]
        relevant = item["relevant_chunk_indices"]
        
        # retrieve with or without rewriting
        if use_rewriting:
            rewrites = rewrite_query(question)
            all_queries = [question] + rewrites
            hits = retrieve_multi(all_queries, k=k)
            retrieved_indices = []
            for chunk, score in hits:
                for i, c in enumerate(chunks):
                    if c == chunk:
                        retrieved_indices.append(i)
                        break
        else:
            retrieved_indices = get_retrieved_indices(question, k=k)
        
        r_at_k = recall_at_k(retrieved_indices, relevant, k)
        rr = reciprocal_rank(retrieved_indices, relevant)
        
        recall_scores.append(r_at_k)
        mrr_scores.append(rr)
        
        # truncate question for display
        q_display = question[:47] + "..." if len(question) > 47 else question
        print(f"{q_display:<50} {r_at_k:>10.2f} {rr:>8.3f}")
    
    print("-" * 70)
    print(f"{'MEAN':<50} {sum(recall_scores)/len(recall_scores):>10.2f} "
          f"{sum(mrr_scores)/len(mrr_scores):>8.3f}")
    
    return {
        "recall_at_k": sum(recall_scores) / len(recall_scores),
        "mrr": sum(mrr_scores) / len(mrr_scores)
    }

In [24]:
print("WITHOUT query rewriting:")
scores_baseline = run_eval(eval_set, k=10, use_rewriting=False)

print("\n\nWITH query rewriting:")
scores_rewriting = run_eval(eval_set, k=10, use_rewriting=True)

print("\n\nIMPROVEMENT:")
print(f"Recall@10: {scores_baseline['recall_at_k']:.2f} → "
      f"{scores_rewriting['recall_at_k']:.2f} "
      f"(+{scores_rewriting['recall_at_k'] - scores_baseline['recall_at_k']:.2f})")
print(f"MRR:       {scores_baseline['mrr']:.3f} → "
      f"{scores_rewriting['mrr']:.3f} "
      f"(+{scores_rewriting['mrr'] - scores_baseline['mrr']:.3f})")

WITHOUT query rewriting:
Running eval | k=10 | query_rewriting=False

Question                                             Recall@k       RR
----------------------------------------------------------------------
What approach does PARS use?                             1.00    0.143
What does PARS stand for?                                1.00    0.200
What is EI?                                              1.00    1.000
How does PARS improve the performance?                   0.00    0.000
What are the compression results for ResNet18?           1.00    0.250
What are the evaluation results of AlexNet usin...       1.00    1.000
What was observed during the rank search proces...       1.00    0.333
What was the top-1 and top-5 accuracies of VGG-...       0.00    0.000
How is the over-fitting avoided on validation s...       1.00    0.500
What is the compression ratio constraint for VG...       0.00    0.000
----------------------------------------------------------------------
MEAN   

In [25]:
def is_specific_technical_query(question):
    # questions with proper nouns, acronyms, numbers = already specific
    # send directly to retrieval, skip rewriting
    specific_signals = [
        any(word.isupper() and len(word) > 1 for word in question.split()),  # acronyms: PARS, EI
        any(char.isdigit() for char in question),   # numbers: ResNet18, 3.5 FLOP
        question.count(" ") < 6,                     # short = probably specific
    ]
    return any(specific_signals)


In [26]:
# Split your eval set by question type
specific_questions = [
    "What does PARS stand for?",
    "What are the compression results for ResNet18?",
    "What was the result under 3.5 FLOP compression ratio for ResNet-18?",
    "What was the top-1 and top-5 accuracies of VGG-16 with BN?",
    "What is EI?",
]

vague_questions = [
    "What problem does this paper address?",
    "What approach does PARS use?",
    "How does PARS improve the performance?",
    "How is the over-fitting avoided on validation set?",
]

print("=== SPECIFIC QUESTIONS ===")
for q in specific_questions:
    hits_direct = retrieve(q, k=10)
    rewrites = rewrite_query(q)
    hits_rewritten = retrieve_multi([q] + rewrites, k=10)
    
    score_direct = hits_direct[0][1] if hits_direct else 0
    score_rewritten = hits_rewritten[0][1] if hits_rewritten else 0
    print(f"Q: {q[:50]}")
    print(f"  Direct top score:   {score_direct:.3f}")
    print(f"  Rewritten top score:{score_rewritten:.3f}")
    print()

print("=== VAGUE QUESTIONS ===")
for q in vague_questions:
    hits_direct = retrieve(q, k=10)
    rewrites = rewrite_query(q)
    hits_rewritten = retrieve_multi([q] + rewrites, k=10)
    
    score_direct = hits_direct[0][1] if hits_direct else 0
    score_rewritten = hits_rewritten[0][1] if hits_rewritten else 0
    print(f"Q: {q[:50]}")
    print(f"  Direct top score:   {score_direct:.3f}")
    print(f"  Rewritten top score:{score_rewritten:.3f}")
    print()

=== SPECIFIC QUESTIONS ===
Q: What does PARS stand for?
  Direct top score:   0.634
  Rewritten top score:0.834

Q: What are the compression results for ResNet18?
  Direct top score:   0.801
  Rewritten top score:0.863

Q: What was the result under 3.5 FLOP compression rat
  Direct top score:   0.812
  Rewritten top score:0.868

Q: What was the top-1 and top-5 accuracies of VGG-16 
  Direct top score:   0.697
  Rewritten top score:0.840

Q: What is EI?
  Direct top score:   0.482
  Rewritten top score:0.810

=== VAGUE QUESTIONS ===
Q: What problem does this paper address?
  Direct top score:   0.523
  Rewritten top score:0.845

Q: What approach does PARS use?
  Direct top score:   0.689
  Rewritten top score:0.865

Q: How does PARS improve the performance?
  Direct top score:   0.688
  Rewritten top score:0.879

Q: How is the over-fitting avoided on validation set?
  Direct top score:   0.706
  Rewritten top score:0.814



In [27]:
def score_of_correct_chunk(question, relevant_indices, use_rewriting=False, k=10):
    """
    Returns the score and rank of the correct chunk specifically.
    This is what actually matters — not the top score overall.
    """
    if use_rewriting:
        rewrites = rewrite_query(question)
        hits = retrieve_multi([question] + rewrites, k=k)
    else:
        hits = retrieve(question, k=k)
    
    # find where the correct chunk landed
    for rank, (chunk, score) in enumerate(hits, start=1):
        for i, c in enumerate(chunks):
            if c == chunk and i in relevant_indices:
                return rank, score   # found it — return its rank and score
    
    return None, 0.0   # correct chunk not in top-k at all

print(f"{'Question':<45} {'Direct':>16} {'Rewritten':>16}")
print(f"{'':45} {'rank | score':>16} {'rank | score':>16}")
print("-" * 80)

for item in eval_set:
    q = item["question"]
    relevant = item["relevant_chunk_indices"]
    
    rank_d, score_d = score_of_correct_chunk(q, relevant, use_rewriting=False)
    rank_r, score_r = score_of_correct_chunk(q, relevant, use_rewriting=True)
    
    rank_d_str  = f"#{rank_d} ({score_d:.3f})"  if rank_d  else "NOT FOUND"
    rank_r_str  = f"#{rank_r} ({score_r:.3f})"  if rank_r  else "NOT FOUND"
    
    q_display = q[:42] + "..." if len(q) > 42 else q
    print(f"{q_display:<45} {rank_d_str:>16} {rank_r_str:>16}")

Question                                                Direct        Rewritten
                                                  rank | score     rank | score
--------------------------------------------------------------------------------
What approach does PARS use?                        #7 (0.620)       #2 (0.843)
What does PARS stand for?                           #5 (0.566)       #3 (0.765)
What is EI?                                         #1 (0.482)        NOT FOUND
How does PARS improve the performance?               NOT FOUND       #3 (0.865)
What are the compression results for ResNe...       #4 (0.771)       #4 (0.822)
What are the evaluation results of AlexNet...       #1 (0.793)       #9 (0.793)
What was observed during the rank search p...       #3 (0.827)       #6 (0.827)
What was the top-1 and top-5 accuracies of...        NOT FOUND       #3 (0.779)
How is the over-fitting avoided on validat...       #2 (0.683)        NOT FOUND
What is the compression ratio constrain

In [28]:
# check every eval item - print the actual chunk content
for item in eval_set:
    print(f"\nQ: {item['question']}")
    print(f"Labeled chunks: {item['relevant_chunk_indices']}")
    for idx in item['relevant_chunk_indices']:
        print(f"\n  chunk[{idx}]:")
        print(f"  {chunks[idx][:300]}")
    print("=" * 60)


Q: What approach does PARS use?
Labeled chunks: [1, 3, 20]

  chunk[1]:
  group of methods decomposes the pre-trained neural network weights through low-rank matrix/tensor decomposition and replaces the original layers with lightweight factorized layers. A main drawback of the technique is that it demands a great amount of time and effort to select the best ranks of tenso

  chunk[3]:
  NP-hard [ 2]. The problem is complicated because inﬂuence of layers to accuracy of the compressed NNs is different and they should be processed individually. An improper decomposition rank for one layer can even lead to a degradation of the model, which can make it inapplicable. This paper proposes 

  chunk[20]:
  good set of ranks in a relatively small number of iter- ations. To this end, a Bayesian optimization procedure [30] is used to solve the constrained optimization (Problem 2) and to ﬁnd the best set of ranks. It builds a surrogate function for the objective and iteratively samples an appropri

In [29]:
def smart_ask(question, show_chunks=False):
    """
    Routes between direct retrieval and query rewriting
    based on question type — learned from eval results.
    """
    # signals that mean "already specific — skip rewriting"
    has_numbers     = any(char.isdigit() for char in question)
    has_model_names = any(name in question for name in 
                         ["ResNet", "VGG", "AlexNet", "MobileNet", 
                          "FLOP", "top-1", "top-5"])
    has_result_words = any(word in question.lower() for word in 
                          ["result", "accuracy", "performance", 
                           "compression ratio", "score"])
    
    is_specific = has_numbers or has_model_names or has_result_words
    
    if is_specific:
        print("  [routing → direct retrieval]")
        hits = retrieve(question, k=10)
    else:
        print("  [routing → query rewriting]")
        rewrites = rewrite_query(question)
        hits = retrieve_multi([question] + rewrites, k=10)
    
    if show_chunks:
        for i, (chunk, score) in enumerate(hits[:3]):
            print(f"  Rank {i+1} | {score:.3f} | {chunk[:100]}...")
    
    return answer(question, hits)

In [30]:
def run_eval_smart(eval_set, k=10):
    recall_scores = []
    mrr_scores = []
    
    print(f"{'Question':<45} {'Recall':>8} {'RR':>8} {'Route':>12}")
    print("-" * 76)
    
    for item in eval_set:
        question = item["question"]
        relevant = item["relevant_chunk_indices"]
        
        # same routing logic as smart_ask
        has_numbers      = any(char.isdigit() for char in question)
        has_model_names  = any(n in question for n in 
                              ["ResNet", "VGG", "AlexNet", "FLOP", "top-1"])
        has_result_words = any(w in question.lower() for w in 
                              ["result", "accuracy", "compression ratio"])
        is_specific = has_numbers or has_model_names or has_result_words
        
        if is_specific:
            hits = retrieve(question, k=k)
            route = "direct"
        else:
            rewrites = rewrite_query(question)
            hits = retrieve_multi([question] + rewrites, k=k)
            route = "rewriting"
        
        # get retrieved indices
        retrieved_indices = []
        for chunk, score in hits:
            for i, c in enumerate(chunks):
                if c == chunk:
                    retrieved_indices.append(i)
                    break
        
        r_at_k = recall_at_k(retrieved_indices, relevant, k)
        rr     = reciprocal_rank(retrieved_indices, relevant)
        recall_scores.append(r_at_k)
        mrr_scores.append(rr)
        
        q_display = question[:42] + "..." if len(question) > 42 else question
        print(f"{q_display:<45} {r_at_k:>8.2f} {rr:>8.3f} {route:>12}")
    
    print("-" * 76)
    print(f"{'MEAN':<45} "
          f"{sum(recall_scores)/len(recall_scores):>8.2f} "
          f"{sum(mrr_scores)/len(mrr_scores):>8.3f}")

run_eval_smart(eval_set)

Question                                        Recall       RR        Route
----------------------------------------------------------------------------
What approach does PARS use?                      1.00    0.200    rewriting
What does PARS stand for?                         1.00    0.500    rewriting
What is EI?                                       0.00    0.000    rewriting
How does PARS improve the performance?            1.00    0.333    rewriting
What are the compression results for ResNe...     1.00    0.250       direct
What are the evaluation results of AlexNet...     1.00    1.000       direct
What was observed during the rank search p...     1.00    0.333       direct
What was the top-1 and top-5 accuracies of...     0.00    0.000       direct
How is the over-fitting avoided on validat...     0.00    0.000    rewriting
What is the compression ratio constraint f...     0.00    0.000       direct
----------------------------------------------------------------------------

In [31]:
# Cell 20 - imports and state definition
from typing import TypedDict, List, Tuple, Optional
from langgraph.graph import StateGraph, END

class AgentState(TypedDict):
    """
    The shared memory that flows through every node.
    Every node reads from this and writes back to it.
    """
    # inputs
    question:        str              # the original user question
    
    # routing decision
    question_type:   str              # "specific" or "conceptual"
    
    # retrieval
    search_query:    str              # current query being retrieved
    retrieved:       List[Tuple]      # all chunks found so far
    retrieval_count: int              # how many retrieval loops so far
    
    # reasoning
    reasoning:       str              # LLM's assessment of what it found
    needs_more:      bool             # should we retrieve again?
    missing_info:    str              # what to search for next
    
    # output
    answer:          Optional[str]    # final answer, None until ready

print("State defined.")

State defined.


In [32]:
# Cell 21 - Router node
def router_node(state: AgentState) -> AgentState:
    question = state["question"]
    
    # conceptual signals override specificity
    comparison_words = any(w in question.lower() for w in
                      ["compare", "why", "how does", "what makes",
                       "difference", "better", "worse", "advantage",
                       "approach", "method", "explain", "what is",
                       "what were", "which datasets", "describe"])
    
    # specific signals — numbers and result lookups
    has_numbers      = any(char.isdigit() for char in question)
    has_result_words = any(w in question.lower() for w in
                          ["result", "accuracy", "top-1", "top-5",
                           "compression ratio", "score", "stand for"])
    
    # conceptual questions take priority even if they contain model names
    if comparison_words and not has_numbers:
        question_type = "conceptual"
    elif has_numbers or has_result_words:
        question_type = "specific"
    else:
        question_type = "conceptual"   # default to rewriting when unsure
    
    print(f"\n[Router] Question type: {question_type}")
    print(f"[Router] Question: {question}")
    
    return {
        **state,
        "question_type":   question_type,
        "search_query":    question,
        "retrieved":       [],
        "retrieval_count": 0,
        "needs_more":      True,
        "answer":          None,
    }

print("Router node defined.")

Router node defined.


In [33]:
# Cell 22 - Retriever node
def retriever_node(state: AgentState) -> AgentState:
    """
    Takes the current search_query.
    Retrieves chunks — direct or with rewriting based on question_type.
    Adds new chunks to state without replacing previous ones.
    """
    query         = state["search_query"]
    question_type = state["question_type"]
    existing      = state["retrieved"]
    count         = state["retrieval_count"]
    
    print(f"\n[Retriever] Loop #{count + 1}")
    print(f"[Retriever] Searching for: {query}")
    
    if question_type == "specific":
        new_hits = retrieve(query, k=5)
        print(f"[Retriever] Strategy: direct retrieval")
    else:
        rewrites = rewrite_query(query)
        new_hits = retrieve_multi([query] + rewrites, k=5)
        print(f"[Retriever] Strategy: query rewriting")
    
    # merge new hits with existing — deduplicate by chunk text
    seen_texts = {chunk for chunk, _ in existing}
    merged = list(existing)
    for chunk, score in new_hits:
        if chunk not in seen_texts:
            merged.append((chunk, score))
            seen_texts.add(chunk)
    
    print(f"[Retriever] Total chunks so far: {len(merged)}")
    
    return {
        **state,
        "retrieved":       merged,
        "retrieval_count": count + 1,
    }

print("Retriever node defined.")

Retriever node defined.


In [34]:
# Cell 23 - Reasoner node
REASONER_PROMPT = """You are evaluating whether retrieved context is sufficient 
to answer a research paper question.

Question: {question}

Retrieved context:
{context}

Respond in exactly this format with no extra text:
SUFFICIENT: yes or no
REASONING: one sentence explaining why
MISSING: if no, write a SHORT search query (3-6 words max) to find what is missing. if yes, write none
ANSWER: if yes, write the complete answer. if no, write none"""

def reasoner_node(state: AgentState) -> AgentState:
    """
    Reads question + all retrieved chunks.
    Asks LLM: is this enough to answer?
    If yes  → sets needs_more=False, drafts answer
    If no   → sets needs_more=True, identifies what to search next
    """
    question  = state["question"]
    retrieved = state["retrieved"]
    count     = state["retrieval_count"]
    
    print(f"\n[Reasoner] Evaluating {len(retrieved)} chunks...")
    
    # build context from top chunks (sort by score, take top 6)
    top_chunks = sorted(retrieved, key=lambda x: x[1], reverse=True)[:6]
    context    = "\n\n---\n\n".join(chunk for chunk, _ in top_chunks)
    
    resp = ollama.chat(
        model=LLM_MODEL,
        messages=[{
            "role": "user",
            "content": REASONER_PROMPT.format(
                question=question,
                context=context
            )
        }],
        options={"temperature": 0.1}
    )
    
    raw = resp["message"]["content"]
    print(f"[Reasoner] Raw response:\n{raw}")
    
    # parse the structured response
    lines = {line.split(":")[0].strip(): ":".join(line.split(":")[1:]).strip()
             for line in raw.strip().split("\n") if ":" in line}
    
    sufficient  = lines.get("SUFFICIENT", "no").lower().strip() == "yes"
    reasoning   = lines.get("REASONING",  "")
    missing     = lines.get("MISSING",    "")
    draft       = lines.get("ANSWER",     "none")
    
    # safety valve — don't loop more than 3 times
    if count >= 3:
        print("[Reasoner] Max loops reached — forcing answer")
        sufficient = True
    
    print(f"[Reasoner] Sufficient: {sufficient}")
    if not sufficient:
        print(f"[Reasoner] Missing: {missing}")
    
    return {
        **state,
        "reasoning":    reasoning,
        "needs_more":   not sufficient,
        "missing_info": missing,
        # if sufficient, store draft answer — Answer node will refine it
        "answer":       draft if sufficient and draft != "none" else None,
        # update search query to what's missing for next loop
        "search_query": missing if not sufficient else state["search_query"],
    }

print("Reasoner node defined.")

Reasoner node defined.


In [35]:
# Cell 24 - Answer node
ANSWER_PROMPT = """You are a research assistant answering questions 
about a specific paper.

Answer the question using ONLY the context below.
If the answer is not in the context, say exactly: "Not found in document."
Be specific — include numbers, method names, and results where available.

Context:
{context}

Question: {question}

Answer:"""

def answer_node(state: AgentState) -> AgentState:
    """
    Takes all retrieved chunks accumulated across all loops.
    Generates the final grounded answer.
    """
    question  = state["question"]
    retrieved = state["retrieved"]
    
    print(f"\n[Answer] Generating final answer from {len(retrieved)} chunks")
    
    # use top 6 chunks by score
    top_chunks = sorted(retrieved, key=lambda x: x[1], reverse=True)[:6]
    context    = "\n\n---\n\n".join(chunk for chunk, _ in top_chunks)
    
    resp = ollama.chat(
        model=LLM_MODEL,
        messages=[{
            "role": "user",
            "content": ANSWER_PROMPT.format(
                context=context,
                question=question
            )
        }],
        options={"temperature": 0.1}
    )
    
    final_answer = resp["message"]["content"]
    
    return {
        **state,
        "answer": final_answer
    }

print("Answer node defined.")

Answer node defined.


In [36]:
# Cell 25 - Build the graph
def should_retrieve_again(state: AgentState) -> str:
    """
    Conditional edge function.
    Returns the name of the next node to go to.
    """
    if state["needs_more"]:
        return "retriever"    # loop back
    else:
        return "answer"       # we have enough, generate final answer

# build the graph
graph = StateGraph(AgentState)

# add nodes
graph.add_node("router",    router_node)
graph.add_node("retriever", retriever_node)
graph.add_node("reasoner",  reasoner_node)
graph.add_node("answer",    answer_node)

# add edges
graph.set_entry_point("router")              # always start here

graph.add_edge("router",    "retriever")     # router always goes to retriever
graph.add_edge("retriever", "reasoner")      # retriever always goes to reasoner

graph.add_conditional_edges(                 # reasoner decides what's next
    "reasoner",
    should_retrieve_again,
    {
        "retriever": "retriever",            # needs_more=True  → loop back
        "answer":    "answer"                # needs_more=False → finish
    }
)

graph.add_edge("answer", END)                # answer always ends the graph

# compile
agent = graph.compile()
print("Graph compiled. Ready to run.")

Graph compiled. Ready to run.


In [37]:
# Cell 26 - Run the agent
def run_agent(question):
    print("=" * 60)
    print(f"QUESTION: {question}")
    print("=" * 60)
    
    # initialise state with just the question
    # all other fields get filled in by nodes
    initial_state = {
        "question":        question,
        "question_type":   "",
        "search_query":    "",
        "retrieved":       [],
        "retrieval_count": 0,
        "reasoning":       "",
        "needs_more":      True,
        "missing_info":    "",
        "answer":          None,
    }
    
    final_state = agent.invoke(initial_state)
    
    print("\n" + "=" * 60)
    print("FINAL ANSWER:")
    print("=" * 60)
    print(final_state["answer"])
    print(f"\n[Retrieved {len(final_state['retrieved'])} chunks "
          f"in {final_state['retrieval_count']} loop(s)]")
    
    return final_state

# test with a simple question first
result = run_agent("What does PARS stand for?")

QUESTION: What does PARS stand for?

[Router] Question type: specific
[Router] Question: What does PARS stand for?

[Retriever] Loop #1
[Retriever] Searching for: What does PARS stand for?
[Retriever] Strategy: direct retrieval
[Retriever] Total chunks so far: 5

[Reasoner] Evaluating 5 chunks...
[Reasoner] Raw response:
SUFFICIENT: yes
REASONING: The question "What does PARS stand for?" is directly answered in the text as "Proxy-based Automatic tensor Rank Selection method".
MISSING: none
ANSWER: Proxy-based Automatic tensor Rank Selection method
[Reasoner] Sufficient: True

[Answer] Generating final answer from 5 chunks

FINAL ANSWER:
Proxy-based Automatic tensor Rank Selection

[Retrieved 5 chunks in 1 loop(s)]


In [ ]:
result = run_agent(
    "What datasets were used to evaluate PARS and what were the final accuracy results on each?"
)

QUESTION: What datasets were used to evaluate PARS and what were the final accuracy results on each?

[Router] Question type: conceptual
[Router] Question: What datasets were used to evaluate PARS and what were the final accuracy results on each?

[Retriever] Loop #1
[Retriever] Searching for: What datasets were used to evaluate PARS and what were the final accuracy results on each?
[Retriever] Strategy: query rewriting
[Retriever] Total chunks so far: 5

[Reasoner] Evaluating 5 chunks...
[Reasoner] Raw response:
SUFFICIENT: no
REASONING: The retrieved context does not mention the specific datasets used to evaluate PARS.
MISSING: What datasets were used to evaluate PARS
ANSWER: none
[Reasoner] Sufficient: False
[Reasoner] Missing: What datasets were used to evaluate PARS

[Retriever] Loop #2
[Retriever] Searching for: What datasets were used to evaluate PARS
[Retriever] Strategy: query rewriting
[Retriever] Total chunks so far: 8

[Reasoner] Evaluating 8 chunks...
[Reasoner] Raw respon